# Cell 1 – Imports & environment check

In [21]:
import os
import json
import random
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score, hamming_loss,
    classification_report, multilabel_confusion_matrix
)

import bpr_pipeline as bpr

DATA_ROOT = Path("../data").resolve()
assert DATA_ROOT.exists(), f"Expected data root at {DATA_ROOT}, check your cwd"

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_PATH = DATA_ROOT / "qwen_bpr_lora"       
CHECKPOINT_DIR = DATA_ROOT / "training_checkpoints"

print("Imports OK")
print("DATA_ROOT:", DATA_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

Imports OK
DATA_ROOT: C:\Users\yousu\Downloads\SAP\project\data
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM (GB): 6.4


# Cell 2 – Load train/eval samples

In [22]:
TRAIN_DIR = DATA_ROOT / "processed" / "train"
EVAL_DIR = DATA_ROOT / "processed" / "eval"

TRAIN_SAMPLE_SIZE = 2500
EVAL_SAMPLE_SIZE = 500

train_files = sorted(TRAIN_DIR.glob("*.json"))
eval_files = sorted(EVAL_DIR.glob("*.json"))

print(f"Total train files available: {len(train_files)}")
print(f"Total eval files available:  {len(eval_files)}")

if not train_files or not eval_files:
    raise FileNotFoundError(
        f"No processed files found under {TRAIN_DIR} / {EVAL_DIR}. "
        "Did notebook 5 finish, and are the files actually at these paths?"
    )

random.seed(42)
train_sample_files = random.sample(train_files, min(TRAIN_SAMPLE_SIZE, len(train_files)))
eval_sample_files = random.sample(eval_files, min(EVAL_SAMPLE_SIZE, len(eval_files)))

def load_records(file_list, desc):
    out = []
    for fp in tqdm(file_list, desc=desc):
        with open(fp, "r", encoding="utf-8") as f:
            out.append(json.load(f))
    return out

train_records = load_records(train_sample_files, "Loading train")
eval_records = load_records(eval_sample_files, "Loading eval")

print(f"\nLoaded {len(train_records)} train records, {len(eval_records)} eval records")
print("Example process:", train_records[0]["as-is"]["process_name"])
print("Example task count (as-is):", len(train_records[0]["as-is"]["process_task"]))

Total train files available: 87766
Total eval files available:  21941


Loading eval: 100%|██████████| 500/500 [00:04<00:00, 109.81it/s]


Loaded 2500 train records, 500 eval records
Example process: equipment order
Example task count (as-is): 8


# Cell 3 – Build prompt/completion pairs

In [ ]:
def build_prompt(as_is_record, as_is_metrics):
    return f"""You are a business process redesign assistant. Given an AS-IS process
and its performance metrics, redesign it to reduce cycle time and cost. Apply
Reijers & Mansar best practices where they qualify. Output valid JSON only,
matching the schema: {{"to-be": <process>, "redesignTrace": [<heuristic entries>]}}.

AS-IS process:
{json.dumps(as_is_record, separators=(',', ':'))}

AS-IS metrics:
- Cycle time: {as_is_metrics['cycle_time_minutes']} minutes
- Cost per case: ${as_is_metrics['labor_cost_per_case']}
- Cycle time efficiency: {as_is_metrics['cycle_time_efficiency_percent']}%
"""

def build_completion(combined_record):
    target = {"to-be": combined_record["to-be"], "redesignTrace": combined_record["redesignTrace"]}
    return json.dumps(target, separators=(',', ':'))

print("Prompt/completion builders defined (compact JSON, no indent, to save tokens)")

# Cell 4 – Compute AS-IS metrics

In [24]:
# --- FlowAnalysisService: GraphBuilder / PathEnumerator / calculate_process_metrics ---
MAX_COMPOSITE_DEPTH = 3

def _task_times(task: dict):
    proc = task.get("expected_process_time") or 0
    wait = task.get("expected_waiting_time") or 0
    rework = task.get("expected_rework_time") or 0
    return float(proc), float(wait), float(rework)

def _task_cost(task: dict) -> float:
    proc, _wait, rework = _task_times(task)
    d_i_hours = (proc + rework) / 60.0
    cost = 0.0
    for job_task in task.get("jobTasks", []) or []:
        job = job_task.get("job") or {}
        hourly_rate = float(job.get("hourlyRate") or 0)
        alloc_pct = float(job_task.get("time_allocation_percentage") or 0)
        cost += d_i_hours * hourly_rate * (alloc_pct / 100.0)
    return cost

@dataclass
class TaskInfo:
    task_id: int
    order: int
    proc_time: float
    wait_time: float
    rework_time: float
    cost: float
    child_process_id: Optional[int] = None

def _build_task_index(process_json: dict):
    tasks = {}
    for pt in process_json.get("process_task", []) or []:
        task = pt.get("task")
        child_process_id = pt.get("child_process_id")
        if task is None:
            synthetic_id = -(child_process_id or pt.get("process_task_id"))
            tasks[synthetic_id] = TaskInfo(synthetic_id, pt.get("order", 0), 0.0, 0.0, 0.0, 0.0, child_process_id)
            continue
        proc, wait, rework = _task_times(task)
        tasks[task["task_id"]] = TaskInfo(
            task["task_id"], pt.get("order", 0), proc, wait, rework, _task_cost(task), child_process_id
        )
    return tasks

@dataclass
class Node:
    kind: str
    ref: Optional[int] = None
    label: Optional[str] = None
    terminal_after: bool = False

def _normalize_branch_probs(branches):
    raw = [float(b.get("probability") or 0) for b in branches]
    total = sum(raw)
    if total <= 0:
        n = len(branches) or 1
        return [1.0 / n] * len(branches)
    if abs(total - 1.0) < 1e-9:
        return raw
    return [p / total for p in raw]

def _resolve_branch_target(branch):
    if branch.get("target_task_id") is not None:
        return Node(kind="task", ref=branch["target_task_id"], terminal_after=bool(branch.get("connect_to_end")))
    if branch.get("target_gateway_id") is not None:
        return Node(kind="gateway", ref=branch["target_gateway_id"])
    return Node(kind="end", label=branch.get("end_event_name"))

class GraphBuilder:
    def __init__(self, process_json: dict):
        self.tasks = _build_task_index(process_json)
        self.gateways = process_json.get("gateways", []) or []
        self.gateway_by_id = {g["gateway_pk_id"]: g for g in self.gateways}
        self.gateway_by_after_task = {g["after_task_id"]: g for g in self.gateways if g.get("after_task_id") is not None}
        self._ordered_task_ids = sorted(self.tasks.keys(), key=lambda tid: self.tasks[tid].order)

    def find_start_node(self):
        targeted = set()
        for g in self.gateways:
            for b in g.get("branches", []):
                if b.get("target_gateway_id") is not None:
                    targeted.add(b["target_gateway_id"])
        qualifying = [g for g in self.gateways if g.get("after_task_id") is None and g["gateway_pk_id"] not in targeted]
        if qualifying:
            return Node(kind="gateway", ref=qualifying[0]["gateway_pk_id"])
        if self._ordered_task_ids:
            return Node(kind="task", ref=self._ordered_task_ids[0])
        return Node(kind="end", label=None)

    def next_after_task(self, task_id):
        gw = self.gateway_by_after_task.get(task_id)
        if gw is not None:
            return Node(kind="gateway", ref=gw["gateway_pk_id"])
        order = self.tasks[task_id].order
        later = [tid for tid in self._ordered_task_ids if self.tasks[tid].order > order]
        if later:
            return Node(kind="task", ref=later[0])
        return Node(kind="end", label=None)

    def converge_target(self, gateway):
        if gateway.get("converge_at_task_id") is not None:
            return Node(kind="task", ref=gateway["converge_at_task_id"])
        if gateway.get("converge_at_gateway_id") is not None:
            return Node(kind="gateway", ref=gateway["converge_at_gateway_id"])
        if gateway.get("converge_to_end"):
            return Node(kind="end", label=gateway.get("converge_gateway_name"))
        return Node(kind="end", label=None)

@dataclass
class PathResult:
    probability: float
    task_ids: list = field(default_factory=list)
    end_label: Optional[str] = None

class PathEnumerator:
    def __init__(self, graph, child_processes=None, max_paths=5000):
        self.graph = graph
        self.child_processes = child_processes or {}
        self.max_paths = max_paths
        self.warnings = []

    def enumerate(self):
        start = self.graph.find_start_node()
        results = []
        self._walk(start, 1.0, [], frozenset(), 0, results)
        if not results:
            results.append(PathResult(probability=1.0, task_ids=[]))
        return results

    def _walk(self, node, probability, task_ids, visited_gateways, depth, results):
        if len(results) >= self.max_paths:
            self.warnings.append("max_paths limit reached; enumeration truncated")
            return
        if node.kind == "end":
            results.append(PathResult(probability=probability, task_ids=list(task_ids), end_label=node.label))
            return
        if node.kind == "task":
            task_ids = task_ids + [node.ref]
            nxt = Node(kind="end", label=None) if node.terminal_after else self.graph.next_after_task(node.ref)
            self._walk(nxt, probability, task_ids, visited_gateways, depth, results)
            return
        if node.kind == "gateway":
            gateway = self.graph.gateway_by_id.get(node.ref)
            if gateway is None:
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            if node.ref in visited_gateways:
                self.warnings.append(f"cyclic reference detected at gateway {node.ref}; loop truncated")
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            visited_gateways = visited_gateways | {node.ref}
            gtype = (gateway.get("gateway_type") or "EXCLUSIVE").upper()
            branches = gateway.get("branches", []) or []
            if not branches:
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            if gtype == "EXCLUSIVE":
                probs = _normalize_branch_probs(branches)
                for b, p in zip(branches, probs):
                    self._walk(_resolve_branch_target(b), probability * p, task_ids, visited_gateways, depth, results)
                return
            if gtype == "PARALLEL":
                merged = list(task_ids)
                for b in branches:
                    merged += self._collect_branch_tasks(_resolve_branch_target(b), gateway)
                self._walk(self.graph.converge_target(gateway), probability, merged, visited_gateways, depth, results)
                return
            if gtype == "INCLUSIVE":
                n = len(branches)
                probs = [float(b.get("probability") or 0) for b in branches]
                for mask in range(1, 1 << n):
                    subset = [i for i in range(n) if mask & (1 << i)]
                    subset_p = 1.0
                    for i in range(n):
                        subset_p *= probs[i] if i in subset else (1.0 - probs[i])
                    merged = list(task_ids)
                    for i in subset:
                        merged += self._collect_branch_tasks(_resolve_branch_target(branches[i]), gateway)
                    self._walk(self.graph.converge_target(gateway), probability * subset_p, merged, visited_gateways, depth, results)
                return
            probs = _normalize_branch_probs(branches)
            for b, p in zip(branches, probs):
                self._walk(_resolve_branch_target(b), probability * p, task_ids, visited_gateways, depth, results)

    def _collect_branch_tasks(self, node, owning_gateway, _depth=0):
        collected = []
        converge = self.graph.converge_target(owning_gateway)
        cur = node
        while _depth < 500:
            if cur.kind == "end":
                break
            if cur.kind == "task":
                if converge.kind == "task" and cur.ref == converge.ref:
                    break
                collected.append(cur.ref)
                cur = self.graph.next_after_task(cur.ref)
                _depth += 1
                continue
            if cur.kind == "gateway":
                if converge.kind == "gateway" and cur.ref == converge.ref:
                    break
                self.warnings.append(f"nested gateway {cur.ref} inside a parallel/inclusive branch was not expanded")
                break
        return collected

def _path_metrics(path, tasks):
    pt_k = wt_k = rt_k = c_k = 0.0
    for tid in path.task_ids:
        info = tasks.get(tid)
        if info is None:
            continue
        pt_k += info.proc_time
        wt_k += info.wait_time
        rt_k += info.rework_time
        c_k += info.cost
    return {"PT_k": pt_k, "WT_k": wt_k, "RT_k": rt_k, "D_k": pt_k + wt_k + rt_k, "C_k": c_k}

def calculate_process_metrics(process_json, child_processes=None, _depth=0):
    if _depth > MAX_COMPOSITE_DEPTH:
        return {"cycle_time_minutes": 0.0, "labor_cost_per_case": 0.0, "processing_time_minutes": 0.0,
                "waiting_time_minutes": 0.0, "rework_time_minutes": 0.0,
                "cycle_time_efficiency_percent": 0.0, "paths_evaluated": 0,
                "warnings": ["max composite sub-process depth (3) exceeded; returned zeros"]}

    graph = GraphBuilder(process_json)
    enumerator = PathEnumerator(graph, child_processes=child_processes)
    paths = enumerator.enumerate()
    warnings = list(enumerator.warnings)

    for tid, info in graph.tasks.items():
        if info.child_process_id is not None:
            child = (child_processes or {}).get(info.child_process_id)
            if child is not None:
                cr = calculate_process_metrics(child, child_processes=child_processes, _depth=_depth + 1)
                info.proc_time, info.wait_time = cr["processing_time_minutes"], cr["waiting_time_minutes"]
                info.rework_time, info.cost = cr["rework_time_minutes"], cr["labor_cost_per_case"]
            else:
                warnings.append(f"composite sub-process slot references child_process_id={info.child_process_id} "
                                 f"but no matching JSON was supplied; treated as zero-duration/zero-cost")

    e_ct = e_pt = e_wt = e_rt = e_cost = 0.0
    for path in paths:
        m = _path_metrics(path, graph.tasks)
        e_ct += path.probability * m["D_k"]; e_pt += path.probability * m["PT_k"]
        e_wt += path.probability * m["WT_k"]; e_rt += path.probability * m["RT_k"]
        e_cost += path.probability * m["C_k"]

    cte = (e_pt / e_ct * 100.0) if e_ct > 0 else 0.0
    return {"cycle_time_minutes": round(e_ct, 2), "labor_cost_per_case": round(e_cost, 2),
            "processing_time_minutes": round(e_pt, 2), "waiting_time_minutes": round(e_wt, 2),
            "rework_time_minutes": round(e_rt, 2), "cycle_time_efficiency_percent": round(cte, 2),
            "paths_evaluated": len(paths), "warnings": warnings}


train_as_is_metrics = [calculate_process_metrics(r["as-is"]) for r in tqdm(train_records, desc="Train AS-IS metrics")]
eval_as_is_metrics = [calculate_process_metrics(r["as-is"]) for r in tqdm(eval_records, desc="Eval AS-IS metrics")]

print("Example AS-IS metrics:", train_as_is_metrics[0])

Eval AS-IS metrics: 100%|██████████| 500/500 [00:00<00:00, 1127.15it/s]

Example AS-IS metrics: {'cycle_time_minutes': 1135.8, 'labor_cost_per_case': 210.29, 'processing_time_minutes': 975.0, 'waiting_time_minutes': 36.0, 'rework_time_minutes': 124.8, 'cycle_time_efficiency_percent': 85.84, 'paths_evaluated': 4, 'warnings': []}


# Cell 5 – Load tokenizer

In [25]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", MODEL_NAME)
print("Pad token:", tokenizer.pad_token, "EOS token:", tokenizer.eos_token)

Tokenizer loaded: Qwen/Qwen2.5-0.5B-Instruct
Pad token: <|endoftext|> EOS token: <|im_end|>


# Cell 6 – Tokenize dataset

In [ ]:
MAX_LENGTH = 4096

def _full_token_len(record, as_is_metrics):
    prompt = build_prompt(record["as-is"], as_is_metrics)
    completion = build_completion(record)
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    full_text = prompt_text + completion + tokenizer.eos_token
    return len(tokenizer(full_text, add_special_tokens=False)["input_ids"])

def build_example(record, as_is_metrics):
    prompt = build_prompt(record["as-is"], as_is_metrics)
    completion = build_completion(record)

    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    full_text = prompt_text + completion + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    if len(full_ids) > MAX_LENGTH:
        return None  # skip records that don't fit; logged below

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


# quick sanity check on token-length distribution before committing to MAX_LENGTH
sample_lengths = [_full_token_len(r, m) for r, m in tqdm(
    list(zip(train_records, train_as_is_metrics))[:200], desc="Sampling lengths"
)]
sample_lengths.sort()
print(f"Sample (n={len(sample_lengths)}) token lengths -- "
      f"min: {sample_lengths[0]}, median: {sample_lengths[len(sample_lengths)//2]}, "
      f"p90: {sample_lengths[int(len(sample_lengths)*0.9)]}, max: {sample_lengths[-1]}")
print(f"Would fit under MAX_LENGTH={MAX_LENGTH}: "
      f"{sum(1 for l in sample_lengths if l <= MAX_LENGTH)}/{len(sample_lengths)}")

train_examples = []
skipped = 0
for record, metrics in tqdm(zip(train_records, train_as_is_metrics), total=len(train_records), desc="Tokenizing train"):
    ex = build_example(record, metrics)
    if ex is None:
        skipped += 1
        continue
    train_examples.append(ex)

print(f"Tokenized {len(train_examples)} train examples, skipped {skipped} (exceeded {MAX_LENGTH} tokens)")

assert len(train_examples) > 0, (
    f"All {skipped} examples exceeded MAX_LENGTH={MAX_LENGTH}. "
    "Raise MAX_LENGTH further (check the p90/max printed above) before continuing."
)

train_dataset = Dataset.from_list(train_examples)
print(train_dataset)

# Cell 7 – Load base Qwen model

In [27]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Model loaded:", MODEL_NAME)
print("Model dtype:", next(model.parameters()).dtype)
print("Model device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Model dtype: torch.float16
Model device: cuda:0


# Cell 8 – Attach LoRA adapter

In [28]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


# Cell 9 – Set training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,  # PEFT-wrapped model signature inspection is unreliable; we pass exact columns ourselves
)

print("Training arguments set")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

# Cell 10 – Run training

In [ ]:
def data_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)
    input_ids, attention_mask, labels = [], [], []
    for f in features:
        pad_len = max_len - len(f["input_ids"])
        input_ids.append(f["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(f["attention_mask"] + [0] * pad_len)
        labels.append(f["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

assert len(train_dataset) > 0, "train_dataset is empty -- check MAX_LENGTH vs prompt token lengths in Cell 6"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Starting training...")
train_result = trainer.train()

print("Training complete")
print(train_result.metrics)

with open(DATA_ROOT / "training_metrics.json", "w") as f:
    json.dump(train_result.metrics, f, indent=2)

# Cell 11 – Save adapter

In [ ]:
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}")

# Cell 12 – Generate on eval set

In [ ]:
model.eval()
MAX_NEW_TOKENS = 1024
GEN_CHECKPOINT_FILE = DATA_ROOT / "generation_checkpoint.json"

def extract_json(text: str):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

generated_texts = []
truncated_flags = []

for i, (record, metrics) in enumerate(tqdm(zip(eval_records, eval_as_is_metrics), total=len(eval_records), desc="Generating")):
    prompt = build_prompt(record["as-is"], metrics)
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output[0][input_ids.shape[-1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    generated_texts.append(text)
    truncated_flags.append(len(new_tokens) >= MAX_NEW_TOKENS)

    # checkpoint every 50 examples in case this crashes overnight
    if (i + 1) % 50 == 0:
        with open(GEN_CHECKPOINT_FILE, "w") as f:
            json.dump({"completed": i + 1, "generated_texts": generated_texts,
                       "truncated_flags": truncated_flags}, f)

with open(GEN_CHECKPOINT_FILE, "w") as f:
    json.dump({"completed": len(generated_texts), "generated_texts": generated_texts,
               "truncated_flags": truncated_flags}, f)

print(f"Generated {len(generated_texts)} outputs")
print(f"Truncated (hit max_new_tokens): {sum(truncated_flags)}/{len(truncated_flags)}")

# Cell 13 – Parse and validate outputs

In [ ]:
eval_results = []

for record, gen_text, truncated in tqdm(zip(eval_records, generated_texts, truncated_flags),
                                         total=len(eval_records), desc="Parsing/validating"):
    result = {
        "process_code": record["as-is"]["process_code"],
        "parse_ok": False, "schema_ok": False, "truncated": truncated,
        "ground_truth_trace": record["redesignTrace"],
    }
    parsed = extract_json(gen_text)
    if parsed is None or "to-be" not in parsed or "redesignTrace" not in parsed:
        eval_results.append(result)
        continue
    result["parse_ok"] = True

    problems = bpr.validate_record(parsed["to-be"])
    if problems:
        result["validation_problems"] = problems
        eval_results.append(result)
        continue
    result["schema_ok"] = True
    result["predicted_trace"] = parsed["redesignTrace"]
    result["predicted_to_be"] = parsed["to-be"]
    eval_results.append(result)

n = len(eval_results)
print(f"Parse success:        {sum(r['parse_ok'] for r in eval_results)/n:.1%}")
print(f"Schema/validation OK: {sum(r['schema_ok'] for r in eval_results)/n:.1%}")

# Cell 14 – Score cost/time reduction

In [ ]:
def pct_reduction(before, after):
    return 0.0 if before == 0 else (before - after) / before * 100

for record, as_is_m, result in tqdm(zip(eval_records, eval_as_is_metrics, eval_results),
                                     total=len(eval_records), desc="TO-BE metrics"):
    result["as_is_metrics"] = as_is_m
    result["ground_truth_metrics"] = calculate_process_metrics(record["to-be"])
    if result.get("schema_ok"):
        try:
            result["predicted_metrics"] = calculate_process_metrics(result["predicted_to_be"])
        except Exception as exc:
            result["metrics_error"] = str(exc)
            result["schema_ok"] = False

scoreable = [r for r in eval_results if r.get("schema_ok") and "predicted_metrics" in r]
print(f"Scoreable on process metrics: {len(scoreable)}/{n}")

if scoreable:
    for r in scoreable:
        ct_before = r["as_is_metrics"]["cycle_time_minutes"]
        cost_before = r["as_is_metrics"]["labor_cost_per_case"]
        r["cycle_time_reduction_pred"] = pct_reduction(ct_before, r["predicted_metrics"]["cycle_time_minutes"])
        r["cycle_time_reduction_gt"] = pct_reduction(ct_before, r["ground_truth_metrics"]["cycle_time_minutes"])
        r["cost_reduction_pred"] = pct_reduction(cost_before, r["predicted_metrics"]["labor_cost_per_case"])
        r["cost_reduction_gt"] = pct_reduction(cost_before, r["ground_truth_metrics"]["labor_cost_per_case"])

    avg_ct_pred = float(np.mean([r["cycle_time_reduction_pred"] for r in scoreable]))
    avg_ct_gt = float(np.mean([r["cycle_time_reduction_gt"] for r in scoreable]))
    avg_cost_pred = float(np.mean([r["cost_reduction_pred"] for r in scoreable]))
    avg_cost_gt = float(np.mean([r["cost_reduction_gt"] for r in scoreable]))

    print(f"Avg cycle time reduction -- model: {avg_ct_pred:.1f}%  ground truth: {avg_ct_gt:.1f}%  "
          f"(recovery: {avg_ct_pred/avg_ct_gt*100 if avg_ct_gt else 0:.0f}%)")
    print(f"Avg cost reduction       -- model: {avg_cost_pred:.1f}%  ground truth: {avg_cost_gt:.1f}%  "
          f"(recovery: {avg_cost_pred/avg_cost_gt*100 if avg_cost_gt else 0:.0f}%)")
else:
    avg_ct_pred = avg_ct_gt = avg_cost_pred = avg_cost_gt = 0.0
    print("No scoreable records -- model produced no valid TO-BE outputs.")

# Cell 15 – Score precision/recall/F1

In [ ]:
HEURISTIC_NAMES = [
    "parallelism", "task_elimination", "task_automation", "task_composition",
    "case_based_work", "numerical_involvement", "knock_out",
    "resequencing", "trusted_party", "extra_resources",
]

y_true = np.zeros((len(eval_results), len(HEURISTIC_NAMES)), dtype=int)
y_pred = np.zeros((len(eval_results), len(HEURISTIC_NAMES)), dtype=int)

for i, r in enumerate(eval_results):
    gt_trace = {t["heuristicName"]: t["isApplied"] for t in r["ground_truth_trace"]}
    for j, name in enumerate(HEURISTIC_NAMES):
        y_true[i, j] = int(gt_trace.get(name, False))
    if not r.get("schema_ok"):
        continue
    pred_trace = {t["heuristicName"]: t.get("isApplied", False) for t in r["predicted_trace"]}
    for j, name in enumerate(HEURISTIC_NAMES):
        y_pred[i, j] = int(pred_trace.get(name, False))

print(classification_report(y_true, y_pred, target_names=HEURISTIC_NAMES, zero_division=0, digits=3))

precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
hamming_acc = 1 - hamming_loss(y_true, y_pred)
exact_match = accuracy_score(y_true, y_pred)

print(f"Micro-avg -- Precision: {precision_micro:.3f}  Recall: {recall_micro:.3f}  F1: {f1_micro:.3f}")
print(f"Macro-avg -- Precision: {precision_macro:.3f}  Recall: {recall_macro:.3f}  F1: {f1_macro:.3f}")
print(f"Hamming accuracy (per-label):        {hamming_acc:.3f}")
print(f"Exact-match accuracy (whole trace):  {exact_match:.3f}")

matrices = multilabel_confusion_matrix(y_true, y_pred)
print("\nPer-heuristic confusion matrices:")
for name, cm in zip(HEURISTIC_NAMES, matrices):
    tn, fp, fn, tp = cm.ravel()
    print(f"{name:24s}  TP={tp:5d}  FP={fp:5d}  FN={fn:5d}  TN={tn:5d}")

aggregate_metrics = {
    "precision_micro": float(precision_micro), "recall_micro": float(recall_micro), "f1_micro": float(f1_micro),
    "precision_macro": float(precision_macro), "recall_macro": float(recall_macro), "f1_macro": float(f1_macro),
    "hamming_accuracy": float(hamming_acc), "exact_match_accuracy": float(exact_match),
}

# Cell 16 – Save results summary

In [ ]:
summary = {
    "model_name": MODEL_NAME,
    "train_sample_size": len(train_records),
    "eval_sample_size": len(eval_records),
    "training_metrics": train_result.metrics,
    "parse_rate": sum(r["parse_ok"] for r in eval_results) / n,
    "schema_ok_rate": sum(r["schema_ok"] for r in eval_results) / n,
    "scoreable_count": len(scoreable),
    "avg_cycle_time_reduction_pred": avg_ct_pred,
    "avg_cycle_time_reduction_gt": avg_ct_gt,
    "avg_cost_reduction_pred": avg_cost_pred,
    "avg_cost_reduction_gt": avg_cost_gt,
    "aggregate_metrics": aggregate_metrics,
}

# NOTE: named "training_summary.json" (not "run_summary.json") -- that name is
# already used by notebook 5's dataset-prep summary in the same data/ folder
# and would otherwise get silently overwritten.
with open(DATA_ROOT / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 50)
print("RUN COMPLETE -- see data/training_summary.json for full details")
print("=" * 50)
for k, v in summary.items():
    if k != "training_metrics":
        print(f"{k}: {v}")